# Challenge 03 — Fase 1: Data Understanding y Geo-Visualización

**TechLogistics S.A.** | Maestría en Ciencia de Datos - EAFIT | Periodo 2026-1

Esta fase cubre las dos tareas asignadas dentro del Challenge 03:

1. **Tarea 1 — Exploración Geo-Temporal:** visualización espacial de la red de
   sensores agroindustriales (`agro_clean.csv`) codificando NDVI y Humedad.
2. **Tarea 2 — Análisis de Estacionariedad y Windowing:** diagnóstico ADF sobre
   las series de energía (`ener_clean.csv`), con ventana móvil para las series
   no estacionarias y análisis de Drift vs. Random Walk en `Ener_5`.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from statsmodels.tsa.stattools import adfuller, kpss

pd.set_option('display.max_columns', None)


## Tarea 1: Exploración Geo-Temporal

Cargamos `agro_clean.csv`. Cada fila representa una lectura de un sensor
agroindustrial en el Oriente Antioqueño, georreferenciada con `Latitude` y
`Longitude`, y conectada a un `Target_Node` (gateway) dentro de la red mesh.


In [2]:
agro = pd.read_csv('../data/raw/agro_clean.csv')
print(agro.shape)
agro.head()


(2000, 14)


,Agro_1,Agro_2,Agro_3,Agro_4,Agro_5,Agro_6,Agro_7,Agro_8,Agro_9,Agro_10,Latitude,Longitude,Source_Node,Target_Node
0,60.248357,47.797447,65.395554,0.000000,0.478718,2.498349,10.000000,6.432151,1.258741,4.663639,6.203680,-75.400366,4,17
1,60.080940,48.076702,66.143237,25.002075,0.467100,2.473166,10.006672,6.469450,1.106051,4.683748,6.282310,-75.474525,14,21
2,60.623974,48.002378,66.342755,49.941596,0.449259,2.464547,10.013349,6.440262,1.183610,8.353141,6.165190,-75.476937,14,22
3,61.211672,48.267738,66.826015,74.756163,0.439299,2.500284,10.020030,6.511042,1.197917,6.213966,6.131334,-75.468152,8,25
4,60.483063,47.912028,65.703353,99.383693,0.436016,2.564177,10.026716,6.619718,1.200795,8.720976,6.241746,-75.349805,5,26


In [3]:
agro.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Agro_1       2000 non-null   float64
 1   Agro_2       2000 non-null   float64
 2   Agro_3       2000 non-null   float64
 3   Agro_4       2000 non-null   float64
 4   Agro_5       2000 non-null   float64
 5   Agro_6       2000 non-null   float64
 6   Agro_7       2000 non-null   float64
 7   Agro_8       2000 non-null   float64
 8   Agro_9       2000 non-null   float64
 9   Agro_10      2000 non-null   float64
 10  Latitude     2000 non-null   float64
 11  Longitude    2000 non-null   float64
 12  Source_Node  2000 non-null   int64  
 13  Target_Node  2000 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 218.9 KB


### Perfilamiento rápido

Antes de visualizar, confirmamos que no hay nulos ni duplicados evidentes
(según el enunciado, esta versión `clean` no debería tenerlos).


In [4]:
print("Nulos por columna:")
print(agro.isna().sum())
print("\nFilas duplicadas:", agro.duplicated().sum())
print("\nSensores únicos (Source_Node):", agro['Source_Node'].nunique())
print("Gateways únicos (Target_Node):", agro['Target_Node'].nunique())


Nulos por columna:
Agro_1         0
Agro_2         0
Agro_3         0
Agro_4         0
Agro_5         0
Agro_6         0
Agro_7         0
Agro_8         0
Agro_9         0
Agro_10        0
Latitude       0
Longitude      0
Source_Node    0
Target_Node    0
dtype: int64

Filas duplicadas: 0

Sensores únicos (Source_Node): 14
Gateways únicos (Target_Node): 15


### Mapa geoespacial: color = NDVI (Agro_5), tamaño = Humedad (Agro_1)

Usamos `scatter_mapbox` de Plotly Express. Como cada sensor tiene múltiples
lecturas en el tiempo, primero agregamos por `Source_Node` (promedio de NDVI
y Humedad, y su ubicación) para tener **un punto por sensor** en el mapa —
graficar cada lectura individual saturaría el mapa sin aportar señal espacial
adicional.


In [5]:
agro_geo = (
    agro.groupby('Source_Node')
    .agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        NDVI=('Agro_5', 'mean'),
        Humedad=('Agro_1', 'mean'),
        Target_Node=('Target_Node', lambda x: x.mode()[0]),
    )
    .reset_index()
)
agro_geo.head()


,Source_Node,Latitude,Longitude,NDVI,Humedad,Target_Node
0,1,6.201189,-75.400394,1.134482,61.303896,17
1,2,6.201115,-75.395744,1.158621,61.185142,16
2,3,6.198764,-75.404026,1.200254,60.859097,20
3,4,6.201792,-75.394481,1.185135,60.961797,19
4,5,6.202941,-75.402092,1.117531,60.846702,19


In [6]:
fig = px.scatter_mapbox(
    agro_geo,
    lat='Latitude',
    lon='Longitude',
    color='NDVI',
    size='Humedad',
    hover_name='Source_Node',
    hover_data={'Target_Node': True, 'NDVI': ':.3f', 'Humedad': ':.2f'},
    color_continuous_scale='RdYlGn',
    size_max=25,
    zoom=9,
    height=600,
    title='Red de Sensores Agroindustriales — Color: NDVI | Tamaño: Humedad',
)
fig.update_layout(mapbox_style='open-street-map')
fig.update_layout(margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig.write_html('../results/figuras/mapa_agro_ndvi_humedad.html', auto_open=False)
fig.show()


C:\Users\user\AppData\Local\Temp\ipykernel_4888\3566232451.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


### Análisis: ¿hay clustering espacial de biomasa baja?

Para responder de forma objetiva (no solo visual), calculamos el umbral del
cuartil inferior de NDVI y revisamos si los sensores por debajo de ese umbral
se concentran geográficamente (distancia promedio entre ellos vs. distancia
promedio general).


In [7]:
q1_ndvi = agro_geo['NDVI'].quantile(0.25)
bajo_ndvi = agro_geo[agro_geo['NDVI'] <= q1_ndvi].copy()
print(f"Umbral Q1 de NDVI: {q1_ndvi:.3f}")
print(f"Sensores con NDVI bajo (<= Q1): {len(bajo_ndvi)} de {len(agro_geo)}")
bajo_ndvi[['Source_Node', 'Latitude', 'Longitude', 'NDVI', 'Humedad']].sort_values('NDVI')


Umbral Q1 de NDVI: 1.137
Sensores con NDVI bajo (<= Q1): 4 de 14


,Source_Node,Latitude,Longitude,NDVI,Humedad
13,14,6.205431,-75.404362,1.105670,61.389763
9,10,6.200163,-75.392642,1.117056,60.312579
4,5,6.202941,-75.402092,1.117531,60.846702
0,1,6.201189,-75.400394,1.134482,61.303896


**Nota metodológica:** usar distancia euclidiana directa sobre grados de
latitud/longitud es incorrecto — un grado de longitud no equivale a la misma
distancia real que un grado de latitud (la equivalencia depende del coseno de
la latitud), y con pocos puntos ese sesgo puede voltear la conclusión.
Usamos la fórmula de **Haversine** para obtener distancias reales en metros
sobre la superficie terrestre.


In [8]:
from itertools import combinations

def haversine(p1, p2):
    """Distancia en metros entre dos puntos (lat, lon) sobre la Tierra."""
    lat1, lon1 = p1
    lat2, lon2 = p2
    R = 6371000  # radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


### Inspección por pares: ¿todos los sensores de NDVI bajo están agrupados, o solo algunos?

Un promedio agregado de distancias sobre el grupo completo no es una métrica
útil aquí: con solo 4 puntos, un único sensor alejado infla el promedio y
puede ocultar que el resto sí esté agrupado. Por eso miramos directamente la
matriz de distancias par a par entre los sensores de NDVI bajo.


In [9]:
nodos_bajo = list(zip(bajo_ndvi['Source_Node'], zip(bajo_ndvi['Latitude'], bajo_ndvi['Longitude'])))

print(f"{'Nodo A':>8} {'Nodo B':>8} {'Distancia (m)':>15}")
pares = []
for (n1, c1), (n2, c2) in combinations(nodos_bajo, 2):
    d = haversine(c1, c2)
    pares.append((n1, n2, d))
    print(f"{n1:>8} {n2:>8} {d:>15.1f}")

pares_df = pd.DataFrame(pares, columns=['Nodo_A', 'Nodo_B', 'Distancia_m']).sort_values('Distancia_m')
pares_df


  Nodo A   Nodo B   Distancia (m)
       1        5           270.6
       1       10           864.5
       1       14           644.1
       5       10          1089.4
       5       14           373.6
      10       14          1421.8


,Nodo_A,Nodo_B,Distancia_m
0,1,5,270.589413
4,5,14,373.590916
2,1,14,644.105813
1,1,10,864.453923
3,5,10,1089.383127
5,10,14,1421.780978


**Conclusión Tarea 1:** la matriz de distancias por pares confirma un
**clúster compacto entre los nodos 14, 5 y 1** (distancias de ~270 a ~644 m
entre ellos), mientras que el **nodo 10 es un outlier aislado** respecto a
ese grupo (~865-1,420 m de distancia).

**Interpretación de negocio:** la zona donde se ubican los nodos 14, 5 y 1 es
la candidata prioritaria para inversión en infraestructura hídrica (pregunta
de negocio P2 de la Fase 4) — ahí sí hay un patrón geográfico de biomasa
baja, no solo fallas puntuales de sensor. El nodo 10 debe tratarse por
separado: su NDVI bajo parece ser un caso aislado, posiblemente ligado a una
condición local del sensor/cultivo más que a la zona.


## Tarea 2: Análisis de Estacionariedad y Windowing

Cargamos `ener_clean.csv`. El diccionario de datos etiqueta explícitamente:
- `Ener_5-7` (Costo de Gas, Emisiones CO2, etc.): **No Estacionarias**.
- `Ener_8-10` (Frecuencia, Voltaje, Factor de Potencia): **Estacionarias**.

Para `Ener_1-3` (Demanda, Precio, Temperatura) el diccionario solo indica que
tienen correlación alta entre sí, sin pronunciarse sobre estacionariedad —
por eso el ADF/KPSS sobre esas tres variables es exploratorio, no una
verificación de algo ya anunciado.

El test ADF nos permite confirmar empíricamente las expectativas del
diccionario y, de paso, diagnosticar lo que no estaba anunciado.


In [10]:
ener = pd.read_csv('../data/raw/ener_clean.csv')
print(ener.shape)
ener.head()


(2000, 14)


,Ener_1,Ener_2,Ener_3,Ener_4,Ener_5,Ener_6,Ener_7,Ener_8,Ener_9,Ener_10,Latitude,Longitude,Source_Node,Target_Node
0,117.538413,169.976747,31.052273,40.000000,5.035559,999.997980,500.382001,60.007129,109.525783,0.952104,6.772983,-75.450170,106,244
1,123.085137,178.287791,30.367913,40.001201,4.964209,1000.012288,499.156777,60.010171,109.396517,0.963803,6.817829,-76.987525,113,241
2,122.444639,178.064846,29.483741,40.004804,4.885030,1000.185140,499.276252,59.998643,110.560808,0.939290,5.519644,-73.757957,105,233
3,117.595897,175.257364,29.985873,40.010804,5.039720,1000.075905,499.093300,59.994357,109.234773,0.945306,8.788327,-73.611230,105,237
4,123.471177,186.082293,30.359202,40.019198,5.046006,999.714688,499.380771,59.955593,110.202660,0.951658,8.450681,-73.456254,108,206


In [11]:
ener.info()
print("\nNulos por columna:")
print(ener.isna().sum())
print("\nFilas duplicadas:", ener.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Ener_1       2000 non-null   float64
 1   Ener_2       2000 non-null   float64
 2   Ener_3       2000 non-null   float64
 3   Ener_4       2000 non-null   float64
 4   Ener_5       2000 non-null   float64
 5   Ener_6       2000 non-null   float64
 6   Ener_7       2000 non-null   float64
 7   Ener_8       2000 non-null   float64
 8   Ener_9       2000 non-null   float64
 9   Ener_10      2000 non-null   float64
 10  Latitude     2000 non-null   float64
 11  Longitude    2000 non-null   float64
 12  Source_Node  2000 non-null   int64  
 13  Target_Node  2000 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 218.9 KB

Nulos por columna:
Ener_1         0
Ener_2         0
Ener_3         0
Ener_4         0
Ener_5         0
Ener_6         0
Ener_7         0
Ener_8         0
Ener_9         0
Ener_1

### Test ADF sobre todas las series `Ener_1` a `Ener_10`

- **H0:** la serie tiene raíz unitaria → no estacionaria.
- **H1:** la serie es estacionaria.
- Regla: si `p-value < 0.05` → rechazamos H0 → **estacionaria (I(0))**.

Complementamos con **KPSS** (hipótesis nula invertida: H0 = estacionaria) para
tener un diagnóstico cruzado más robusto, especialmente en los casos donde
ADF dé un resultado ambiguo.


In [12]:
def diagnostico_estacionariedad(serie, nombre):
    adf_stat, adf_p, *_ = adfuller(serie.dropna(), autolag='AIC')

    # KPSS puede lanzar warning de interpolación en los p-values extremos; lo ignoramos aquí
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        kpss_stat, kpss_p, *_ = kpss(serie.dropna(), regression='c', nlags='auto')

    adf_concl = 'Estacionaria' if adf_p < 0.05 else 'No Estacionaria'
    kpss_concl = 'No Estacionaria' if kpss_p < 0.05 else 'Estacionaria'

    return {
        'Variable': nombre,
        'ADF_stat': round(adf_stat, 4),
        'ADF_p': round(adf_p, 4),
        'ADF_conclusion': adf_concl,
        'KPSS_stat': round(kpss_stat, 4),
        'KPSS_p': round(kpss_p, 4),
        'KPSS_conclusion': kpss_concl,
        'Coinciden': adf_concl == kpss_concl,
    }

cols_ener = [f'Ener_{i}' for i in range(1, 11)]
resultados = [diagnostico_estacionariedad(ener[c], c) for c in cols_ener]
df_estacionariedad = pd.DataFrame(resultados)
df_estacionariedad


,Variable,ADF_stat,ADF_p,ADF_conclusion,KPSS_stat,KPSS_p,KPSS_conclusion,Coinciden
0,Ener_1,-1.866900e+00,0.3478,No Estacionaria,0.3836,0.0842,Estacionaria,False
1,Ener_2,-1.479100e+00,0.5438,No Estacionaria,0.3701,0.0900,Estacionaria,False
2,Ener_3,-1.928400e+00,0.3188,No Estacionaria,0.3735,0.0886,Estacionaria,False
3,Ener_4,-8.418193e+09,0.0000,Estacionaria,0.0684,0.1000,Estacionaria,True
4,Ener_5,-3.477000e-01,0.9185,No Estacionaria,6.9235,0.0100,No Estacionaria,True
5,Ener_6,9.527000e-01,0.9937,No Estacionaria,6.9870,0.0100,No Estacionaria,True
6,Ener_7,-4.244000e-01,0.9059,No Estacionaria,7.0007,0.0100,No Estacionaria,True
7,Ener_8,-4.603630e+01,0.0000,Estacionaria,0.0489,0.1000,Estacionaria,True
8,Ener_9,-4.535820e+01,0.0000,Estacionaria,0.0438,0.1000,Estacionaria,True
9,Ener_10,-4.371510e+01,0.0000,Estacionaria,0.1114,0.1000,Estacionaria,True


In [13]:
df_estacionariedad.to_csv('../results/diagnostico_estacionariedad_ener.csv', index=False)
no_estacionarias = df_estacionariedad[df_estacionariedad['ADF_conclusion'] == 'No Estacionaria']['Variable'].tolist()
print("Series clasificadas como NO estacionarias (ADF):", no_estacionarias)


Series clasificadas como NO estacionarias (ADF): ['Ener_1', 'Ener_2', 'Ener_3', 'Ener_5', 'Ener_6', 'Ener_7']


### Ventana móvil (50 registros) para las series no estacionarias

Para cada serie no estacionaria calculamos media y varianza móvil con
ventana de 50 registros. Graficarlas todas juntas en la misma escala es
engañoso: `Ener_6` y `Ener_7` tienen rangos de cientos de unidades mientras
`Ener_1-3` se mueven en decenas, así que estas últimas se verían "planas"
por escala, no porque carezcan de tendencia. Por eso usamos subplots
independientes, cada uno con su propio eje.


In [14]:
from plotly.subplots import make_subplots

fig_rolling = make_subplots(
    rows=2, cols=3,
    subplot_titles=no_estacionarias,
)
posiciones = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]

for (col, (row, colpos)) in zip(no_estacionarias, posiciones):
    roll_mean = ener[col].rolling(window=50).mean()
    fig_rolling.add_scatter(
        x=ener.index, y=roll_mean, name=col, row=row, col=colpos, showlegend=False,
    )

fig_rolling.update_layout(height=600, title_text='Media Móvil (ventana=50) — Escala independiente por variable')
fig_rolling.write_html('../results/figuras/media_movil_no_estacionarias.html', auto_open=False)
fig_rolling.show()


### Métrica objetiva de fuerza de tendencia

Para no depender de una lectura visual, calculamos, por cada serie no
estacionaria: el **rango de su media móvil** (máximo - mínimo) dividido entre
la **desviación estándar de la serie completa**. Un ratio alto indica que la
media móvil se desplaza más de lo que varía el ruido normal de la serie
→ tendencia dominante. Un ratio bajo indica que la media móvil apenas se
mueve en comparación con el ruido propio de la serie → comportamiento más
cercano a estacionario alrededor de una tendencia débil.


In [15]:
fuerza_tendencia = []
for col in no_estacionarias:
    roll_mean = ener[col].rolling(window=50).mean().dropna()
    rango_media_movil = roll_mean.max() - roll_mean.min()
    std_serie = ener[col].std()
    ratio = rango_media_movil / std_serie
    fuerza_tendencia.append({'Variable': col, 'Rango_media_movil': round(rango_media_movil, 3),
                              'Std_serie': round(std_serie, 3), 'Ratio_tendencia_ruido': round(ratio, 3)})

df_fuerza = pd.DataFrame(fuerza_tendencia).sort_values('Ratio_tendencia_ruido', ascending=False)
df_fuerza


,Variable,Rango_media_movil,Std_serie,Ratio_tendencia_ruido
5,Ener_7,39.141,11.585,3.379
4,Ener_6,192.276,57.405,3.349
3,Ener_5,20.956,6.474,3.237
0,Ener_1,40.427,14.440,2.800
2,Ener_3,10.116,3.615,2.799
1,Ener_2,61.197,21.898,2.795


**Nota:** el ratio tendencia/ruido resultó similar en orden de magnitud
para las seis series (~2.8 a ~3.4), así que no se usa como criterio de
decisión — la clasificación se basa en la coincidencia o no entre ADF y
KPSS, como se estableció antes.


### Decisión de tratamiento: casos ambiguos

**Marcamos `Ener_1`, `Ener_2` y `Ener_3` como CASOS AMBIGUOS**: ADF y KPSS se
contradicen (`Coinciden = False`), a diferencia de `Ener_5-7` donde ambos
tests coinciden en "No Estacionaria". El ratio tendencia/ruido no ayuda a
desambiguar (resultó similar en las seis series), así que la decisión se basa
únicamente en el patrón de acuerdo/desacuerdo entre ADF y KPSS.

**Decisión adoptada:** por precaución metodológica, tratamos los casos
ambiguos como **No Estacionarias** para todo el análisis posterior
(diferenciación antes de correlación de Pearson, ventana móvil, y cualquier
modelo tipo ARIMA/ARIMAX en fases siguientes). La razón es asimétrica: si una
serie ambigua en realidad es estacionaria y la tratamos como no estacionaria,
en el peor caso perdemos un poco de información al diferenciar de más; pero
si es realmente no estacionaria y la tratamos como estacionaria, el riesgo es
una correlación espuria o un modelo mal especificado — un error más grave
para las conclusiones de negocio. Esta es la postura que se debe citar en el
Informe Técnico al responder la pregunta de validación #1 del checklist.


In [16]:
fig_var = px.line(title='Varianza Móvil (ventana=50) — Series No Estacionarias')
for col in no_estacionarias:
    roll_var = ener[col].rolling(window=50).var()
    fig_var.add_scatter(x=ener.index, y=roll_var, name=f'{col} (varianza móvil)')
fig_var.update_layout(height=500, xaxis_title='Índice temporal', yaxis_title='Varianza')
fig_var.write_html('../results/figuras/varianza_movil_no_estacionarias.html', auto_open=False)
fig_var.show()


### Ener_5 (Costo del Gas): ¿Drift o Random Walk?

- **Drift:** la media móvil crece (o decrece) de forma sostenida y consistente
  en una dirección — hay una tendencia determinística subyacente.
- **Random Walk puro:** la media móvil fluctúa sin una dirección clara, el
  valor de hoy es simplemente el de ayer más un shock aleatorio, sin sesgo
  direccional sistemático.


In [17]:
fig_gas = px.line(
    x=ener.index, y=ener['Ener_5'],
    title='Ener_5 (Costo del Gas) — Serie original',
    labels={'x': 'Índice temporal', 'y': 'Ener_5'},
)
roll_mean_gas = ener['Ener_5'].rolling(window=50).mean()
fig_gas.add_scatter(x=ener.index, y=roll_mean_gas, name='Media móvil (50)', line=dict(color='red', width=3))
fig_gas.update_layout(height=500)
fig_gas.write_html('../results/figuras/ener5_drift_vs_randomwalk.html', auto_open=False)
fig_gas.show()


In [18]:
# Pendiente de la tendencia (regresión lineal simple sobre la media móvil, sin NaN iniciales)
serie_valida = roll_mean_gas.dropna()
x = np.arange(len(serie_valida))
pendiente, intercepto = np.polyfit(x, serie_valida.values, 1)
print(f"Pendiente de la media móvil de Ener_5: {pendiente:.6f}")

if abs(pendiente) > 1e-3:
    print("→ La media móvil muestra una tendencia direccional sostenida: comportamiento consistente con DRIFT.")
else:
    print("→ La media móvil no muestra tendencia direccional clara: comportamiento consistente con RANDOM WALK puro.")


Pendiente de la media móvil de Ener_5: 0.011128
→ La media móvil muestra una tendencia direccional sostenida: comportamiento consistente con DRIFT.


**Conclusión Tarea 2:** con la coincidencia o no entre ADF y KPSS como
criterio, y la pendiente de la media móvil de `Ener_5` confirmando un
comportamiento de Drift, esta es la clasificación final:

**Clasificación final adoptada:**
- **No Estacionarias (evidencia sólida, ADF y KPSS coinciden):** `Ener_5`,
  `Ener_6`, `Ener_7`.
- **No Estacionarias (caso ambiguo, tratadas como tal por precaución):**
  `Ener_1`, `Ener_2`, `Ener_3`.
- **Estacionarias (evidencia sólida, ADF y KPSS coinciden):** `Ener_4`,
  `Ener_8`, `Ener_9`, `Ener_10`.

Esta clasificación es la que se debe usar consistentemente en las Fases 2-4
del taller (diferenciación previa a Pearson/ARIMA, selección de variables
para el modelo ARIMAX, etc.).
